# BAF EDA Gap Matrix

Documented walkthrough for the account-level EDA comparison matrix.

**Policy:** Temporal kNN is used **only** for graph edge construction (`Application—SIMILAR—Application`).

**Primary metric:** PR-AUC on test month 7 (threshold tuned on validation month 6).
**Secondary ops metric:** Business economics (revenue TP, unlock/check FP costs, dual cutoff profit).


## 0. Setup and protocol

### Goal
Fix the experimental protocol before any model runs: time split, leakage rules, all-feature schema, cost placeholders, and output paths.

### Decision rule
All downstream cells must use `train months 0–5`, `valid month 6`, `test month 7`. Graph edges require `neighbor_month < query_month`. No feature subsetting unless an explicit ablation enables it.


In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eda.feature_plots import draw_features_baf, list_baf_feature_columns
from src.graph.property_graph import (
    build_property_graph,
    property_graph_embeddings,
    property_graph_stats,
)
from src.graph.temporal_knn import (
    TemporalKNNConfig,
    build_temporal_knn_graph,
    mean_neighbor_features,
    temporal_knn_stats,
)
from src.modeling.cost_metrics import (
    CostConfig,
    build_cutoff_metrics_table,
    compute_revenue_tp_per_account,
    cost_check_fp_per_account,
    cost_unlock_fp_per_account,
    maximize_profit,
    plot_cutoff_economics,
)
from src.modeling.metrics import best_f1_threshold, calibrate_platt, evaluate_binary_classifier
from src.modeling.train_vanilla import train_vanilla
from src.modeling.xgb_runtime import resolve_xgb_compute
from src.preprocessing.baf_preprocessor import BAFPreprocessor, TimeSplit
from xgboost import XGBClassifier

COST_CONFIG = CostConfig(
    clerk_salary_hkd_annual=400_000,
    hours_per_month=160,
    hours_to_unlock_fp=32,
    hours_to_check_fp=8,
    fraud_volume_proxy_col="intended_balcon_amount",
)

CONFIG = {
    "seed": 42,
    "data_path": REPO_ROOT / "data" / "base.csv",
    "variant_paths": [REPO_ROOT / "data" / f"variant_{i}.csv" for i in range(1, 7)],
    "results_dir": REPO_ROOT / "results" / "eda_matrix",
    "stage1_variant": REPO_ROOT / "data" / "base.csv",
    "knn_k": 20,
    "knn_metric": "cosine",
    "knn_mutual": False,
    "use_smote_stage1": False,
    "use_yeo_johnson": True,
    "recall_at_precision": 0.8,
    "eda_plot_all_features": True,
    "eda_max_plots": None,
    "shap_sample_size": 500,
}

CONFIG["results_dir"].mkdir(parents=True, exist_ok=True)
np.random.seed(CONFIG["seed"])
print("Repo root:", REPO_ROOT)
print("Results:", CONFIG["results_dir"])


## 1. Exploratory Data Analysis (Part A — Colab-style)

### Goal
Inspect **all** BAF features: target stats by month, factor plots (fraud vs non-fraud %), missingness, and numeric correlations before graph construction.

### Decision rule
Large category skew or bin-level separation → candidate signal; include via `BAFPreprocessor` (no manual feature dropping).

### Interpretation
Proxy for blocked fraudulent exposure uses `intended_balcon_amount`; replace with real fraud volume when available.


In [ ]:
DATA_PATH = Path(CONFIG["stage1_variant"])
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"BAF CSV not found at {DATA_PATH}. Place base.csv under data/ or update CONFIG."
    )

df = pd.read_csv(DATA_PATH)
numeric_cols, categorical_cols = list_baf_feature_columns(df)
all_feature_cols = categorical_cols + numeric_cols
print("Shape:", df.shape)
print("Features:", len(all_feature_cols), "categorical:", len(categorical_cols), "numeric:", len(numeric_cols))
print("\nFraud rate overall:", df["fraud_bool"].mean())
print("\nFraud rate by month:")
print(df.groupby("month")["fraud_bool"].agg(["count", "mean"]).rename(columns={"mean": "fraud_rate"}))

fig, ax = plt.subplots(figsize=(8, 3))
df.groupby("month")["fraud_bool"].mean().plot(kind="bar", ax=ax, title="Fraud rate by month")
ax.set_ylabel("fraud_rate")
plt.tight_layout()
plt.show()


In [ ]:
# Colab-style factor plots on all raw BAF columns
plot_cols = all_feature_cols
if CONFIG.get("eda_max_plots"):
    plot_cols = plot_cols[: CONFIG["eda_max_plots"]]
elif not CONFIG.get("eda_plot_all_features", True):
    plot_cols = plot_cols[:10]
draw_features_baf(df, feature_cols=plot_cols)


In [ ]:
# Missingness and -1 markers
missing = df[all_feature_cols].replace(-1, np.nan).isna().mean().sort_values(ascending=False)
print("Top missing / -1 rates:")
print(missing.head(15))

fig, ax = plt.subplots(figsize=(10, 4))
missing.head(20).plot(kind="bar", ax=ax, title="Missing or -1 rate by feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Numeric correlation heatmap (train months only for leakage safety preview)
train_preview = df[df["month"].isin([0, 1, 2, 3, 4, 5])]
num_for_corr = [c for c in numeric_cols if train_preview[c].replace(-1, np.nan).notna().sum() > 0]
corr = train_preview[num_for_corr].replace(-1, np.nan).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
plt.title("Numeric feature correlation (train months 0–5)")
plt.tight_layout()
plt.show()


## 2. Preprocessing (all features)

### Goal
Fit `BAFPreprocessor` on train months only; produce leakage-safe feature matrices using the full BAF schema.

### Decision rule
Preprocessor is fit on months 0–5 only. Validation/test are transform-only.


In [ ]:
split = TimeSplit()
preprocessor = BAFPreprocessor(use_yeo_johnson=CONFIG["use_yeo_johnson"])
train_df, valid_df, test_df = preprocessor.split_by_month(df, split)
preprocessor.fit(train_df)

X_train, y_train = preprocessor.transform_with_target(train_df)
X_valid, y_valid = preprocessor.transform_with_target(valid_df)
X_test, y_test = preprocessor.transform_with_target(test_df)
feature_names = preprocessor.get_feature_names()

features_all = np.vstack([X_train.values, X_valid.values, X_test.values])
labels_all = np.concatenate([y_train.values, y_valid.values, y_test.values])
months_all = np.concatenate([
    train_df["month"].values,
    valid_df["month"].values,
    test_df["month"].values,
])
offsets = {
    "train": (0, len(train_df)),
    "valid": (len(train_df), len(train_df) + len(valid_df)),
    "test": (len(train_df) + len(valid_df), len(features_all)),
}
print("Train/valid/test:", X_train.shape, X_valid.shape, X_test.shape)
print("Preprocessed feature dim:", X_train.shape[1])
print("Sample feature names:", feature_names[:8], "...")


## 3. Anchor XGBoost (no graph)

### Goal
Tabular XGBoost baseline on **all** preprocessed features without any graph structure.

### Decision rule
This is the anchor row. Graph arms must beat this on validation PR-AUC to proceed with graph expansion in Stage 2.


In [ ]:
anchor_out = CONFIG["results_dir"] / "anchor"
anchor_report = train_vanilla(
    DATA_PATH,
    output_dir=anchor_out,
    use_yeo_johnson=CONFIG["use_yeo_johnson"],
    use_smote=CONFIG["use_smote_stage1"],
    prefer_gpu=True,
)
anchor_model = joblib.load(anchor_out / "model.pkl")
test_pred_df = pd.read_csv(anchor_out / "test_predictions.csv")
anchor_test_scores = test_pred_df["score"].values
print("Anchor test PR-AUC:", anchor_report["metrics"]["pr_auc"])


## 4. Feature importances

### Goal
Native XGBoost gain importances (and optional SHAP) on all preprocessed features for the anchor model.

### Decision rule
Export top features to CSV; compare later against graph-augmented models in Stage 1.


In [ ]:
imp = pd.DataFrame({
    "feature": feature_names,
    "importance": anchor_model.feature_importances_,
}).sort_values("importance", ascending=False)
imp["importance_pct"] = imp["importance"] / imp["importance"].sum()

top_n = 30
fig, ax = plt.subplots(figsize=(10, 8))
imp.head(top_n).plot.barh(x="feature", y="importance", ax=ax, legend=False)
ax.invert_yaxis()
ax.set_title(f"Anchor XGBoost — top {top_n} feature importances")
plt.tight_layout()
plt.show()

imp_path = CONFIG["results_dir"] / "feature_importance_anchor.csv"
imp.to_csv(imp_path, index=False)
print("Exported:", imp_path)
print(imp.head(10))


In [ ]:
try:
    import shap

    shap_n = min(CONFIG["shap_sample_size"], len(X_test))
    shap_idx = np.random.choice(len(X_test), shap_n, replace=False)
    X_shap = X_test.iloc[shap_idx]
    explainer = shap.TreeExplainer(anchor_model)
    shap_values = explainer.shap_values(X_shap)
    shap.summary_plot(shap_values, X_shap, feature_names=feature_names, max_display=20, show=False)
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("SHAP skipped:", exc)


## 5. Temporal kNN graph

### Goal
Build `SIMILAR` edges using cosine kNN with temporal constraint `neighbor_month < query_month`.

**Note:** Cosine measures pattern similarity (angle); L2 measures absolute distance. Cosine is the default for mixed numeric + one-hot BAF features.

### Decision rule
If `temporal_violations > 0`, stop and fix the graph builder before training.


In [ ]:
knn_cfg = TemporalKNNConfig(
    k=CONFIG["knn_k"],
    metric=CONFIG["knn_metric"],
    mutual=CONFIG["knn_mutual"],
)
knn_edges = build_temporal_knn_graph(features_all, months_all, labels_all, knn_cfg)
knn_stats = temporal_knn_stats(knn_edges, labels_all, months_all)
print("Temporal kNN stats:", json.dumps(knn_stats, indent=2))
assert knn_stats["temporal_violations"] == 0, "Temporal leakage in kNN graph!"

knn_embeddings = mean_neighbor_features(features_all, knn_edges, len(features_all))
print("kNN embedding shape:", knn_embeddings.shape)


## 6. Property graph

### Goal
Build typed entity nodes (`source`, `device_os`, …) and `HAS_*` edges for interpretable structure.

### Decision rule
Compare property graph stats to kNN graph. Neither replaces anchor tabular features — they provide embeddings for GNN→XGBoost.


In [ ]:
app_nodes, prop_edges, entity_nodes = build_property_graph(df)
prop_stats = property_graph_stats(app_nodes, prop_edges, labels_all)
print("Property graph stats:", json.dumps(prop_stats, indent=2))
print("Entity node sample:")
print(entity_nodes.head())

prop_embeddings = property_graph_embeddings(features_all, prop_edges, len(features_all))
print("Property embedding shape:", prop_embeddings.shape)


## 7. Stage 1 comparison

### Goal
Compare anchor XGBoost vs graph embedding + XGBoost on validation PR-AUC.

### Decision rule
Proceed to Stage 2 only if a graph arm beats anchor on **validation** PR-AUC by a meaningful margin (e.g. ≥ 0.005).


In [ ]:
def train_graph_xgb(name, graph_features, use_smote=False):
    X_tr = np.hstack([X_train.values, graph_features[offsets["train"][0]:offsets["train"][1]]])
    X_va = np.hstack([X_valid.values, graph_features[offsets["valid"][0]:offsets["valid"][1]]])
    X_te = np.hstack([X_test.values, graph_features[offsets["test"][0]:offsets["test"][1]]])
    y_tr, y_va, y_te = y_train.values, y_valid.values, y_test.values

    if use_smote:
        from imblearn.over_sampling import SMOTE
        smote = SMOTE(sampling_strategy=0.5, random_state=CONFIG["seed"])
        X_tr, y_tr = smote.fit_resample(X_tr, y_tr)

    compute = resolve_xgb_compute(prefer_gpu=True)
    model = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=CONFIG["seed"],
        tree_method=compute["tree_method"],
        device=compute["device"],
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    valid_scores = model.predict_proba(X_va)[:, 1]
    test_scores = model.predict_proba(X_te)[:, 1]
    calibrator = calibrate_platt(y_va, valid_scores)
    test_cal = calibrator.model.predict_proba(test_scores.reshape(-1, 1))[:, 1]
    threshold = best_f1_threshold(y_va, valid_scores)
    metrics = evaluate_binary_classifier(y_te, test_scores, test_cal, threshold)
    return {"name": name, "metrics": metrics, "test_scores": test_scores, "model": model}


stage1_rows = [
    {"graph": "none", "model": "XGBoost_tabular", "metrics": anchor_report["metrics"], "test_scores": anchor_test_scores},
]
for label, emb in [("temporal_kNN", knn_embeddings), ("property", prop_embeddings)]:
    stage1_rows.append(train_graph_xgb(f"{label}_GNN_to_XGBoost", emb))

stage1_df = pd.DataFrame([
    {
        "graph": r.get("graph", r["name"].replace("_GNN_to_XGBoost", "")),
        "model": r.get("model", "GNN_to_XGBoost"),
        "pr_auc": r["metrics"]["pr_auc"],
        "recall": r["metrics"]["recall"],
        "brier_calibrated": r["metrics"]["brier_calibrated"],
        "alert_yield_pct": r["metrics"]["alert_yield_pct"],
    }
    for r in stage1_rows
])
print(stage1_df)
stage1_df.to_csv(CONFIG["results_dir"] / "stage1_results.csv", index=False)


## 8. Business cost model (Part C)

### Goal
Translate model scores into operational economics: revenue from true positives, unlock/check costs for false positives, and optimal dual BLOCK/ALERT cutoffs.

### Assumptions
- `revenue_tp_per_account = total_fraud_volume_train / n_fraud_accounts_train` (proxy column: `intended_balcon_amount`)
- Unlock cost: incorrectly blocked legitimate user (churn, support)
- Check cost: analyst review for alerted-but-not-blocked users
- Placeholder HKD salary and hours — replace with real ops data

### Decision rule
Run on anchor test scores first; repeat for Stage 1 graph winner if it beats anchor.


In [ ]:
revenue_tp = compute_revenue_tp_per_account(
    train_df,
    volume_col=COST_CONFIG.fraud_volume_proxy_col,
)
cost_unlock = cost_unlock_fp_per_account(COST_CONFIG)
cost_check = cost_check_fp_per_account(COST_CONFIG)

cost_assumptions = {
    "clerk_salary_hkd_annual": COST_CONFIG.clerk_salary_hkd_annual,
    "hours_per_month": COST_CONFIG.hours_per_month,
    "hours_to_unlock_fp": COST_CONFIG.hours_to_unlock_fp,
    "hours_to_check_fp": COST_CONFIG.hours_to_check_fp,
    "fraud_volume_proxy_col": COST_CONFIG.fraud_volume_proxy_col,
    "revenue_tp_per_account_hkd": revenue_tp,
    "cost_unlock_fp_hkd": cost_unlock,
    "cost_check_fp_hkd": cost_check,
    "note": "Proxy for blocked fraudulent exposure; replace with real fraud volume when available.",
}
assumptions_path = CONFIG["results_dir"] / "cost_assumptions.json"
assumptions_path.write_text(json.dumps(cost_assumptions, indent=2))
print(json.dumps(cost_assumptions, indent=2))


In [ ]:
cutoff_metrics = build_cutoff_metrics_table(
    y_test.values,
    anchor_test_scores,
    revenue_tp=revenue_tp,
    cost_unlock_fp=cost_unlock,
    cost_check_fp=cost_check,
)
cutoff_path = CONFIG["results_dir"] / "cutoff_economics.csv"
cutoff_metrics.to_csv(cutoff_path, index=False)

optimal = maximize_profit(cutoff_metrics, cost_check_fp=cost_check, config=COST_CONFIG)
alert_band = max(optimal["alert_predicted"] - optimal["block_predicted"], 0)
optimal["expected_profit_hkd"] = optimal.pop("max_profit")
optimal_path = CONFIG["results_dir"] / "optimal_cutoffs.json"
optimal_path.write_text(json.dumps(optimal, indent=2))

fig, ax = plt.subplots(figsize=(10, 5))
plot_cutoff_economics(cutoff_metrics, ax=ax)
plt.tight_layout()
plt.show()

print("Optimal cutoffs:", json.dumps(optimal, indent=2))
print("Exported:", cutoff_path, optimal_path)


## 9. Stage gates — Stages 2 and 4

### Goal
Gated ablations: Stage 2 (preprocess × imbalance) and Stage 4 (6 BAF variants) run only after Stage 1 review.

### Decision rule
Skip unless graph arm beats anchor on validation PR-AUC.


In [ ]:
RUN_STAGE2 = False
stage2_results = []

if RUN_STAGE2:
    for use_yj in [True, False]:
        for use_smote in [False, True]:
            report = train_vanilla(
                DATA_PATH,
                output_dir=CONFIG["results_dir"] / f"anchor_yj{use_yj}_smote{use_smote}",
                use_yeo_johnson=use_yj,
                use_smote=use_smote,
            )
            stage2_results.append({
                "arm": "anchor",
                "use_yeo_johnson": use_yj,
                "use_smote": use_smote,
                "pr_auc": report["metrics"]["pr_auc"],
                "recall": report["metrics"]["recall"],
            })
    stage2_df = pd.DataFrame(stage2_results)
    stage2_df.to_csv(CONFIG["results_dir"] / "stage2_ablations.csv", index=False)
    print(stage2_df)
else:
    print("Stage 2 skipped — enable RUN_STAGE2 after Stage 1 gate.")


In [ ]:
RUN_STAGE4 = False
variant_rows = []

if RUN_STAGE4:
    for vpath in CONFIG["variant_paths"]:
        if not vpath.exists():
            print("Skip missing:", vpath)
            continue
        report = train_vanilla(
            vpath,
            output_dir=CONFIG["results_dir"] / vpath.stem,
            use_yeo_johnson=CONFIG["use_yeo_johnson"],
            use_smote=CONFIG["use_smote_stage1"],
        )
        variant_rows.append({"variant": vpath.stem, "pr_auc": report["metrics"]["pr_auc"]})
    variant_df = pd.DataFrame(variant_rows)
    variant_df.to_csv(CONFIG["results_dir"] / "stage4_variants.csv", index=False)
    print(variant_df.describe())
else:
    print("Stage 4 skipped — enable RUN_STAGE4 after champion is frozen.")


## 10. Conclusions and exports

### Goal
Persist graph stats, champion config, feature importances, and cost summary for downstream synthetic-env and paper work.

### Interpretation
If no graph arm beats anchor, document as a negative result. Cost model provides secondary threshold guidance alongside PR-AUC.


In [ ]:
graph_stats = {
    "temporal_knn": knn_stats,
    "property_graph": prop_stats,
    "knn_policy": "graph_edges_only_no_tabular_enrichment",
    "n_raw_features": len(all_feature_cols),
    "n_preprocessed_features": len(feature_names),
    "config": {k: str(v) for k, v in CONFIG.items()},
    "cost_assumptions": cost_assumptions,
}
(CONFIG["results_dir"] / "graph_stats.json").write_text(json.dumps(graph_stats, indent=2))

best_row = stage1_df.sort_values("pr_auc", ascending=False).iloc[0]
champion = {
    "graph": best_row["graph"],
    "model": best_row["model"],
    "pr_auc_test": float(best_row["pr_auc"]),
    "stage1_complete": True,
    "optimal_block_cutoff": optimal["block_cutoff"],
    "optimal_alert_cutoff": optimal["alert_cutoff"],
    "expected_profit_hkd": optimal["expected_profit_hkd"],
}
(CONFIG["results_dir"] / "champion_config.json").write_text(json.dumps(champion, indent=2))

print("Exported:", CONFIG["results_dir"])
print("Champion:", champion)
